根据 case_x 变量，遍历该病例下所有时间点检查；
读取每个检查目录里的 with_data_images 或 without_data_images 文件夹中的图片；
按“一行 6 张”展示；
每张图片下方显示：

1.所属检查号，例如 20230426000263
2.图片文件名

你后续只需要改 case_x 和 image_folder_name 两个变量即可。

In [ ]:
import os
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import math

# =========================
# 1. 参数设置
# =========================
root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\fetal_hydro_dataset_3_report_合并 copy\raw")

case_x = "case_0012"                 # 可修改，例如 "case_0002"
image_folder_name = "without_data_images"   # 可修改为 "without_data_images"

cols = 5   # 一行显示几张

# 支持的图片后缀
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


# =========================
# 2. 收集指定 case 下所有时间点图片
# =========================
def collect_images_for_case(case_dir: Path, image_folder_name: str):
    """
    遍历 case_dir 下所有 phase（如 1prenatal/2postnatal/3postoperative）
    以及每个 phase 下所有检查号目录，收集其中 image_folder_name 文件夹里的图片。
    
    返回:
        image_infos: list of dict
            每个元素包含:
            {
                "img_path": Path,
                "exam_id": 检查号目录名,
                "phase": phase目录名,
                "img_name": 图片文件名
            }
    """
    image_infos = []

    if not case_dir.exists():
        print(f"[错误] case 路径不存在: {case_dir}")
        return image_infos

    # 遍历 1prenatal / 2postnatal / 3postoperative
    for phase_dir in sorted(case_dir.iterdir()):
        if not phase_dir.is_dir():
            continue

        # 遍历每个检查号目录
        for exam_dir in sorted(phase_dir.iterdir()):
            if not exam_dir.is_dir():
                continue

            target_img_dir = exam_dir / image_folder_name
            if not target_img_dir.exists() or not target_img_dir.is_dir():
                continue

            # 收集该检查号目录下的图片
            for img_path in sorted(target_img_dir.iterdir()):
                if img_path.is_file() and img_path.suffix.lower() in img_exts:
                    image_infos.append({
                        "img_path": img_path,
                        "exam_id": exam_dir.name,
                        "phase": phase_dir.name,
                        "img_name": img_path.name
                    })

    return image_infos


# =========================
# 3. 展示图片
# =========================
def show_images(image_infos, cols=5, figsize_per_col=3, figsize_per_row=3.8):
    """
    按网格展示图片，每张图下方显示：
    exam_id
    img_name
    """
    n = len(image_infos)
    if n == 0:
        print("没有找到图片。")
        return

    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * figsize_per_col, rows * figsize_per_row))
    
    # 当只有一行或一列时，统一转成二维处理
    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]

    axes_flat = [ax for row in axes for ax in row]

    for ax, info in zip(axes_flat, image_infos):
        img_path = info["img_path"]
        exam_id = info["exam_id"]
        img_name = info["img_name"]

        try:
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)
            ax.axis("off")
            ax.set_title(f"{exam_id}\n{img_name}", fontsize=9)
        except Exception as e:
            ax.axis("off")
            ax.set_title(f"{exam_id}\n{img_name}\n[读取失败]", fontsize=9)
            print(f"[读取失败] {img_path}: {e}")

    # 多余子图隐藏
    for ax in axes_flat[len(image_infos):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


# =========================
# 4. 主流程
# =========================
case_dir = root_dir / case_x
image_infos = collect_images_for_case(case_dir, image_folder_name)

print(f"病例: {case_x}")
print(f"图片文件夹名: {image_folder_name}")
print(f"共找到图片数: {len(image_infos)}")

show_images(image_infos, cols=cols)